In [1]:
import json
from sklearn.metrics import cohen_kappa_score
# Round 1
annotated_ali = "data_annotated_20_ali.json"
annotated_wilder = "data_annotated_20_wilder.json"

with open(annotated_ali, 'r', encoding='utf-8') as file:
    ali = json.load(file)


with open(annotated_wilder, 'r', encoding='utf-8') as file:
    wilder = json.load(file)

In [2]:
TYPES = ["Lexical", "Syntactic", "Semantic", "Vagueness", "Incompleteness", "Referential"]

for t in TYPES:
    a = [ali[i][t] for i in range(len(ali))]
    w = [wilder[i][t] for i in range(len(wilder))]

    agree = sum(x == y for x, y in zip(a, w)) / len(ali)
    k = cohen_kappa_score(a, w)
    print(f"{t:<15} kappa={k:6.3f}  agreement={agree:.2%} ali_pos={sum(a):2d}  wilder_pos={sum(w):2d}")

Lexical         kappa=   nan  agreement=100.00% ali_pos= 0  wilder_pos= 0
Syntactic       kappa=-0.053  agreement=90.00% ali_pos= 1  wilder_pos= 1
Semantic        kappa=   nan  agreement=100.00% ali_pos= 0  wilder_pos= 0
Vagueness       kappa= 0.375  agreement=80.00% ali_pos= 4  wilder_pos= 4
Incompleteness  kappa=-0.231  agreement=60.00% ali_pos= 5  wilder_pos= 3
Referential     kappa=   nan  agreement=100.00% ali_pos= 0  wilder_pos= 0


/opt/miniconda3/envs/dialogue/lib/python3.10/site-packages/sklearn/metrics/_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/opt/miniconda3/envs/dialogue/lib/python3.10/site-packages/sklearn/metrics/_classification.py:897: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
/opt/miniconda3/envs/dialogue/lib/python3.10/site-packages/sklearn/metrics/_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/opt/miniconda3/envs/dialogue/lib/python3.10/site-packages/sklearn/metrics/_classification.py:897: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expect

In [3]:
# align by id instead of trusting file order
ali_by_id = {r["id"]: r for r in ali}
wilder_by_id = {r["id"]: r for r in wilder}
ids = sorted(set(ali_by_id) & set(wilder_by_id))
disagreements = {t: [i for i in ids if ali_by_id[i][t] != wilder_by_id[i][t]] for t in TYPES}
disagreements

{'Lexical': [],
 'Syntactic': [1, 17],
 'Semantic': [],
 'Vagueness': [5, 15, 18, 19],
 'Incompleteness': [1, 4, 5, 6, 12, 14, 15, 19],
 'Referential': []}

In [4]:
for i in ids:
    diffs = [t for t in TYPES if bool(ali_by_id[i][t]) != bool(wilder_by_id[i][t])]
    if not diffs:
        continue
    print(f"id: {i}")
    print(f"requirement: {ali_by_id[i]['description']}")
    for t in diffs:
        who = "Ali" if ali_by_id[i][t] else "Wilder"
        print(f"  - {t}: True by {who}")
    for name, rec in (("Ali", ali_by_id[i]), ("Wilder", wilder_by_id[i])):
        note = (rec.get("notes") or rec.get("note") or "").strip()
        if note:
            print(f"  note ({name}): {note}")
    print()

id: 1
requirement: The system shall implement procedures for monitoring log-in attempts and reporting discrepancies.
  - Syntactic: True by Ali
  - Incompleteness: True by Wilder
  note (Ali): [monitoring log-in attempts] and [reporting discrepancies] OR monitoring [log-in attempts and reporting discrepancies]

id: 4
requirement: The system shall implement procedures for safeguarding passwords.
  - Incompleteness: True by Ali

id: 5
requirement: The system shall support implementing policies and procedures to address security incidents.
  - Vagueness: True by Ali
  - Incompleteness: True by Ali
  note (Ali): address is vague

id: 6
requirement: The system shall identify and respond to suspected or known security incidents.
  - Incompleteness: True by Ali
  note (Ali): how to respond to? incomplete.

id: 12
requirement: The system shall support technical policies and procedures that allow access to electronic protected health information (ePHI) only for authorized users and authorized s